# Notebook 02 — FanGraphs Data Pull

Source: `fangraphs.com` leaderboards via `pybaseball` (handles Cloudflare), with direct URL fallback  
Rate limit: 1.5 s between requests, browser-like User-Agent  

This notebook pulls three datasets for the 2025 season:

1. Hitting leaderboard: advanced stats: wRC+, WAR, wOBA, BABIP, batted-ball profile (GB/FB/LD%)  
2. Pitching leaderboard: FIP, xFIP, SIERA, WAR, GB%, K%, BB%  
3. Park factors: single-year and 5-year rolling factors by team  

It also builds a FanGraphs to MLBAM ID crosswalk using normalized name matching, which is the bridge table for merging FanGraphs data with MLB Stats API and Statcast data downstream.

FanGraphs uses its own `fg_id` (IDfg) internally. The MLBAM ID (`mlbam_id`) is the universal join key across all other sources.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from scrapers.fangraphs import (
    get_fg_hitting, get_fg_pitching, get_park_factors, build_fg_mlbam_crosswalk
)
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

## 1. FanGraphs Hitting Leaderboard

**Key columns:**
- `wRC+`: Weighted Runs Created Plus. Park- and league-adjusted; 100 = average. The best single "how good is this hitter" number.
- `WAR`: Wins Above Replacement (FanGraphs version, fWAR).
- `wOBA`: Weighted On-Base Average. Run-value-weighted version of OBP. This fills the gap left by the MLB Stats API (which doesn't provide wOBA).
- `BABIP`: Batting Average on Balls In Play. Useful for spotting luck-driven outliers.
- `GB%` / `FB%` / `LD%`: Batted ball profile. Ground ball, fly ball, and line drive rates.

In [ ]:
fg_hitting = get_fg_hitting(2025)
print(f"Shape: {fg_hitting.shape}")
print(f"\nDtypes:\n{fg_hitting.dtypes}")
print(f"\nNull rates:\n{fg_hitting.isnull().mean().round(4)}")
print(f"\nSample (10 rows):")
fg_hitting.head(10)

## 2. FanGraphs Pitching Leaderboard

**Key columns:**
- `FIP`: Fielding Independent Pitching. Estimates ERA from K, BB, HR, HBP only. Strips out defense.
- `xFIP`: Expected FIP. Same as FIP but normalizes HR/FB rate to the league average (~10%). More predictive than FIP for future performance.
- `SIERA`: Skill-Interactive ERA. The most sophisticated ERA estimator — accounts for batted ball types and their interaction with K/BB rates.
- `WAR`: Pitcher fWAR.
- `GB%`: Ground ball rate. High-GB pitchers suppress HR and induce double plays.
- `K%` / `BB%`: Strikeout and walk rates (as proportions of PA faced).

In [ ]:
fg_pitching = get_fg_pitching(2025)
print(f"Shape: {fg_pitching.shape}")
print(f"\nDtypes:\n{fg_pitching.dtypes}")
print(f"\nNull rates:\n{fg_pitching.isnull().mean().round(4)}")
print(f"\nSample (10 rows):")
fg_pitching.head(10)

## 3. Park Factors

Park factors quantify how much a team's home ballpark inflates or suppresses run scoring relative to a neutral park.

- `park_factor_basic`: single-season (1yr) factor. 100 = neutral. >100 = hitter-friendly. <100 = pitcher-friendly.
- `park_factor_5yr`: 5-year rolling average. More stable and preferred for projections since single-year factors are noisy.

In [ ]:
park_factors = get_park_factors(2025)
print(f"Shape: {park_factors.shape}")
print(f"\nDtypes:\n{park_factors.dtypes}")
print(f"\nNull rates:\n{park_factors.isnull().mean().round(4)}")
print(f"\nAll 30 teams:")
park_factors

## 4. FanGraphs → MLBAM ID Crosswalk

The crosswalk joins FanGraphs data to the MLBAM player universe using a two-pass name matching strategy:

1. Pass 1 — Exact normalized name: Strip accents (José→Jose, Ramón→Ramon), remove suffixes (Jr., Sr., II), lowercase, remove punctuation. Match on full name.
2. Pass 2 — Last name + team: For any remaining unmatched, try matching on last name + team abbreviation to handle first-name variants.

Potential systematic mismatches:
- Mid-season trades: FanGraphs assigns traded players to `"- - -"` (multiple teams). The MLB API only shows the current team, so pass-2 (last name + team) won't match these. Pass-1 (name only) handles most of them.
- Accent characters: FanGraphs strips accents; the MLB API preserves them. Our `unicodedata` normalization handles this.
- Name suffixes: "Ronald Acuña Jr." vs "Ronald Acuna" — suffix removal handles this.

In [ ]:
# Load the MLBAM players roster from Step 3
mlbam_players = pd.read_parquet(os.path.join("..", "data", "raw_mlb_api_players.parquet"))

# Build crosswalk from hitting leaderboard
xwalk_hit = build_fg_mlbam_crosswalk(fg_hitting, mlbam_players)

total = len(xwalk_hit)
matched = (xwalk_hit["match_status"] == "matched").sum()
unmatched = total - matched

print(f"=== Hitting Crosswalk ===")
print(f"Total FG hitters: {total}")
print(f"Matched to MLBAM: {matched} ({matched/total*100:.1f}%)")
print(f"Unmatched:         {unmatched} ({unmatched/total*100:.1f}%)")

if unmatched > 0:
    print(f"\nUnmatched rows:")
    print(xwalk_hit[xwalk_hit["match_status"] == "unmatched"][["fg_id", "name", "team"]].to_string(index=False))

In [ ]:
# Build crosswalk from pitching leaderboard too
xwalk_pit = build_fg_mlbam_crosswalk(fg_pitching, mlbam_players)

total_p = len(xwalk_pit)
matched_p = (xwalk_pit["match_status"] == "matched").sum()
unmatched_p = total_p - matched_p

print(f"=== Pitching Crosswalk ===")
print(f"Total FG pitchers: {total_p}")
print(f"Matched to MLBAM:  {matched_p} ({matched_p/total_p*100:.1f}%)")
print(f"Unmatched:          {unmatched_p} ({unmatched_p/total_p*100:.1f}%)")

if unmatched_p > 0:
    print(f"\nUnmatched rows:")
    print(xwalk_pit[xwalk_pit["match_status"] == "unmatched"][["fg_id", "name", "team"]].to_string(index=False))

In [ ]:
# Combine both crosswalks into a single unified table (dedup by fg_id)
xwalk_combined = pd.concat([xwalk_hit, xwalk_pit], ignore_index=True)
xwalk_combined = xwalk_combined.drop_duplicates(subset=["fg_id"], keep="first")

total_c = len(xwalk_combined)
matched_c = (xwalk_combined["match_status"] == "matched").sum()

print(f"=== Combined Crosswalk ===")
print(f"Total unique fg_ids: {total_c}")
print(f"Matched: {matched_c} ({matched_c/total_c*100:.1f}%)")
print(f"Unmatched: {total_c - matched_c}")

# Flag mid-season trade players (team = "- - -")
trade_players = xwalk_combined[xwalk_combined["team"] == "- - -"]
if len(trade_players) > 0:
    print(f"\n=== Players on multiple teams (traded mid-season): {len(trade_players)} ===")
    print(trade_players[["fg_id", "mlbam_id", "name", "team", "match_status"]].to_string(index=False))

## 5. Null Rate Summary (all DataFrames)

In [ ]:
null_summary = pd.DataFrame({
    "fg_hitting": fg_hitting.isnull().mean().round(4),
    "fg_pitching": fg_pitching.isnull().mean().round(4),
    "park_factors": park_factors.isnull().mean().round(4),
    "crosswalk": xwalk_combined.isnull().mean().round(4),
}).fillna("—")
print("=== Null Rate Summary ===")
null_summary

## 6. Data Quality Notes

1. wOBA from FanGraphs fills the MLB API gap — the MLB Stats API doesn't provide wOBA, but FanGraphs does. We'll use the crosswalk to map it onto the MLBAM-keyed hitting data downstream.

3. FanGraphs team abbreviations — FG uses 2-3 letter abbreviations (NYY, LAD, etc.) while the MLB API uses full team names. The crosswalk bridges this gap through player-level matching rather than team-level matching.

4. Traded players — FanGraphs lists players who were traded mid-season under team `"- - -"`. These still match via Pass 1 (name-only matching).

## 7. Save to Parquet

In [ ]:
data_dir = os.path.join("..", "data")
os.makedirs(data_dir, exist_ok=True)

fg_hitting.to_parquet(os.path.join(data_dir, "raw_fangraphs_hitting.parquet"), index=False)
fg_pitching.to_parquet(os.path.join(data_dir, "raw_fangraphs_pitching.parquet"), index=False)
park_factors.to_parquet(os.path.join(data_dir, "park_factors.parquet"), index=False)
xwalk_combined.to_parquet(os.path.join(data_dir, "fg_mlbam_crosswalk.parquet"), index=False)

print("Saved:")
for f in ["raw_fangraphs_hitting.parquet", "raw_fangraphs_pitching.parquet",
          "park_factors.parquet", "fg_mlbam_crosswalk.parquet"]:
    path = os.path.join(data_dir, f)
    size_kb = os.path.getsize(path) / 1024
    print(f"  {f} — {size_kb:.1f} KB")